# LeetCode 1690 — Stone Game VII bằng Python

Link đề: https://leetcode.com/problems/stone-game-vii/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming, Game Theory

Skeleton LeetCode:

```python
class Solution:
    def stoneGameVII(self, stones: List[int]) -> int:
        ...
```

Lộ trình: **đệ quy minimax thuần** (mũ, chỉ cho `n` nhỏ) → thêm **tổng tiền tố** + **ghi nhớ** theo đoạn `(i, j)` (`O(n²)`) → **DP khoảng bottom-up** không đệ quy → **mảng 1 chiều cuộn** tiết kiệm bộ nhớ `O(n)`. Điểm mấu chốt: định nghĩa `dp[i][j]` = "hiệu điểm tốt nhất của **người sắp đi**" trên đoạn `stones[i..j]`.

## 1. Mục tiêu học

1. Biến trò chơi "Alice tối đa hóa hiệu, Bob tối thiểu hóa hiệu" thành **một** công thức đối xứng: mỗi người tối đa hóa (điểm mình − điểm đối thủ) từ lượt mình trở đi.
2. Thiết kế trạng thái **DP khoảng** `dp[i][j]` trên đoạn còn lại và viết công thức chuyển `dp[i][j] = max(S(i+1..j) − dp[i+1][j], S(i..j−1) − dp[i][j−1])`.
3. Dùng **tổng tiền tố** để tính tổng đoạn trong `O(1)`.
4. Chọn **thứ tự lặp** đúng cho bottom-up (độ dài tăng dần, hoặc `i` giảm dần) và nén về **mảng 1 chiều**.
5. Nhận biết giới hạn của `lru_cache`/đệ quy trong Python với `n = 1000` (độ sâu đệ quy, chi phí gọi hàm).

## 2. Đề bài & ví dụ

Alice và Bob lần lượt chơi, **Alice đi trước**. Có `n` viên đá xếp thành **một hàng**. Mỗi lượt, người chơi bỏ **viên ngoài cùng bên trái hoặc bên phải**, và nhận điểm bằng **tổng giá trị các viên còn lại** trên hàng. Hết đá thì người nhiều điểm hơn thắng.

Bob biết mình luôn thua, nên Bob muốn **tối thiểu hóa hiệu điểm** (Alice − Bob); Alice muốn **tối đa hóa** hiệu đó. Cho `stones`, trả về hiệu điểm Alice − Bob khi cả hai chơi tối ưu.

**Ví dụ 1:** `stones = [5,3,1,4,2]` → `6`.
- Alice bỏ 2, được `5+3+1+4 = 13`. Còn `[5,3,1,4]`.
- Bob bỏ 5, được `3+1+4 = 8`. Còn `[3,1,4]`.
- Alice bỏ 3, được `1+4 = 5`. Còn `[1,4]`.
- Bob bỏ 1, được `4`. Còn `[4]`.
- Alice bỏ 4, được `0`. Hiệu `18 − 12 = 6`.

**Ví dụ 2:** `stones = [7,90,5,1,100,10,10,2]` → `122`.

**Ràng buộc:**
- `n == stones.length`, `2 ≤ n ≤ 1000`
- `1 ≤ stones[i] ≤ 1000`

In [1]:
import random
import time
import sys
from functools import lru_cache
from itertools import accumulate
from typing import List

examples = [
    ([5, 3, 1, 4, 2], 6),
    ([7, 90, 5, 1, 100, 10, 10, 2], 122),
]
for stones, want in examples:
    print(f"stones={stones} -> kỳ vọng {want}")

stones=[5, 3, 1, 4, 2] -> kỳ vọng 6
stones=[7, 90, 5, 1, 100, 10, 10, 2] -> kỳ vọng 122


## 3. Kiến thức nền

### 3.1 Tổng tiền tố (prefix sum)

`pre[k] = stones[0] + … + stones[k−1]`, `pre[0] = 0`. Khi đó tổng đoạn `stones[i..j]` (bao gồm hai đầu) là `pre[j+1] − pre[i]`, tính trong `O(1)`.

In [2]:
stones = [5, 3, 1, 4, 2]
pre = [0] + list(accumulate(stones))
print("pre =", pre)
S = lambda i, j: pre[j + 1] - pre[i] if i <= j else 0
print("tổng stones[1..3] =", S(1, 3), "(= 3+1+4)")
print("tổng đoạn rỗng S(3, 2) =", S(3, 2))

pre = [0, 5, 8, 9, 13, 15]
tổng stones[1..3] = 8 (= 3+1+4)
tổng đoạn rỗng S(3, 2) = 0


### 3.2 Trò chơi tổng-không và "hiệu từ góc nhìn người đi"

Trong trò chơi hai người mà một người muốn **lớn** `A − B` và người kia muốn **nhỏ** `A − B` (⇔ lớn `B − A`), ta định nghĩa:

> `f(state)` = giá trị lớn nhất của (điểm người sắp đi − điểm người kia) tính **từ state trở về sau**.

Nếu người đi chọn nước được `g` điểm và chuyển sang `state'`, thì từ đó đối thủ đạt `f(state')` theo góc nhìn của họ ⇒ với người đi là `g − f(state')`. Vậy `f(state) = max (g − f(state'))`. Một công thức dùng chung cho **cả hai** người — không cần tách hàm max/min.

In [3]:
# Minh họa trên trò chơi nhỏ: mỗi lượt lấy 1 hoặc 2 từ một đống, điểm = số lấy được
@lru_cache(maxsize=None)
def f_pile(k):
    if k == 0:
        return 0
    return max(g - f_pile(k - g) for g in (1, 2) if g <= k)
print([f_pile(k) for k in range(8)])

[0, 1, 2, 1, 0, 1, 2, 1]


### 3.3 DP khoảng (interval DP) và thứ tự lặp

Trạng thái là đoạn `[i, j]`. `dp[i][j]` phụ thuộc vào `dp[i+1][j]` (đoạn ngắn hơn, bắt đầu muộn hơn) và `dp[i][j−1]` (đoạn ngắn hơn). Hai cách lặp hợp lệ:
- **theo độ dài** `len = 2..n`, rồi `i` từ 0;
- **`i` giảm dần** từ `n−1` về 0, `j` tăng dần từ `i+1` — khi tính `dp[i][j]`, hàng `i+1` đã xong và `dp[i][j−1]` vừa tính ở bước trước.

In [4]:
n = 4
print("thứ tự theo i giảm dần, j tăng dần:")
print([(i, j) for i in range(n - 1, -1, -1) for j in range(i + 1, n)])

thứ tự theo i giảm dần, j tăng dần:
[(2, 3), (1, 2), (1, 3), (0, 1), (0, 2), (0, 3)]


### 3.4 Giới hạn đệ quy trong Python

Mặc định `sys.getrecursionlimit()` khoảng 1000. DP đệ quy trên đoạn `[0, n−1]` có thể sâu tới `n` tầng. Với `n = 1000` sẽ sát giới hạn; ngoài ra `lru_cache` với `~5·10⁵` trạng thái khá chậm. Vì vậy bản nộp nên là **bottom-up**.

In [5]:
print("recursion limit hiện tại:", sys.getrecursionlimit())

recursion limit hiện tại: 1000


## 4. Ý tưởng nền tảng / trực giác

Quan sát quan trọng: **điểm nhận được = tổng các viên còn lại** sau khi bỏ. Nếu đoạn hiện tại là `[i, j]`:
- bỏ trái (`stones[i]`) ⇒ được `S(i+1, j)`,
- bỏ phải (`stones[j]`) ⇒ được `S(i, j−1)`.

Tính tay đoạn 2 viên `[a, b]`: bỏ `a` được `b`, sau đó đối thủ bỏ `b` được 0 ⇒ hiệu `b`. Bỏ `b` được `a` ⇒ hiệu `a`. Vậy `dp = max(a, b)`.

Với ví dụ 1, thử nghĩ "tham lam: bỏ viên nhỏ hơn để giữ tổng lớn": Alice bỏ 2 (nhỏ hơn 5) — trùng với lời giải. Nhưng tham lam không luôn đúng (mục 13). Ta cần DP.

In [6]:
def two_stones(a, b):
    left_first = b - 0      # bỏ a, được b; đối thủ bỏ b được 0
    right_first = a - 0
    return max(left_first, right_first)
print(two_stones(3, 8), two_stones(10, 2))

8 10


## 5. Cách 1 — Brute Force / baseline: đệ quy minimax thuần

**Ý tưởng:** thử cả hai lựa chọn ở mỗi lượt, đệ quy trên **danh sách còn lại**, không ghi nhớ. Mỗi lần gọi tính lại `sum()` của danh sách.

`best(arr) = max( sum(arr[1:]) − best(arr[1:]), sum(arr[:-1]) − best(arr[:-1]) )`, `best([]) = 0` (và `best([x]) = 0` vì bỏ viên cuối được 0).

In [7]:
def stone_vii_brute(stones):
    def best(arr):
        if len(arr) <= 1:
            return 0
        left = sum(arr[1:]) - best(arr[1:])
        right = sum(arr[:-1]) - best(arr[:-1])
        return max(left, right)
    return best(list(stones))

for stones, want in examples:
    print(stone_vii_brute(stones), "kỳ vọng", want)

6 kỳ vọng 6
122 kỳ vọng 122


**Độ phức tạp:** cây đệ quy có `2^{n−1}` lá, mỗi nút tốn `O(n)` để cắt list và tính tổng ⇒ `O(n · 2ⁿ)`. Space `O(n²)` trên stack (mỗi tầng giữ một list).

**Nút thắt:** cùng một đoạn `[i, j]` bị tính lại vô số lần (bỏ trái rồi phải ≡ bỏ phải rồi trái). Chỉ có `n²/2` đoạn khác nhau ⇒ ghi nhớ.

In [8]:
random.seed(0)
for n in [10, 14, 17]:
    st = [random.randint(1, 1000) for _ in range(n)]
    t0 = time.perf_counter(); stone_vii_brute(st)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=10: 0.000s
n=14: 0.002s
n=17: 0.018s


## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái:** đoạn còn lại luôn là một **đoạn liên tiếp** `[i, j]` của mảng ban đầu (chỉ bỏ ở hai đầu). Gọi

> `dp[i][j]` = hiệu (điểm người sắp đi − điểm người kia) lớn nhất trên đoạn `stones[i..j]`.

**Công thức:**

`dp[i][j] = max( S(i+1, j) − dp[i+1][j],  S(i, j−1) − dp[i][j−1] )`, với `dp[i][i] = 0`.

**Đáp án:** `dp[0][n−1]`.

**Vì sao "Bob tối thiểu hóa hiệu" khớp công thức?** Bob tối thiểu `A − B` ⇔ tối đa `B − A` — đúng là "hiệu từ góc nhìn người đi". Cả hai cùng dùng một hàm `dp`.

**Invariant khi lặp bottom-up:** khi tính `dp[i][j]`, mọi đoạn **con thật sự** của `[i, j]` đã có giá trị đúng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Top-down: ghi nhớ theo `(i, j)` + tổng tiền tố

Từng đoạn:
1. `pre` = tổng tiền tố.
2. `dfs(i, j)`: nếu `i == j` trả về 0.
3. Hai lựa chọn: `pre[j+1] − pre[i+1] − dfs(i+1, j)` và `pre[j] − pre[i] − dfs(i, j−1)`.
4. `@lru_cache` ghi nhớ; xóa cache sau khi dùng để giải phóng bộ nhớ.

In [9]:
def stone_vii_memo(stones):
    n = len(stones)
    pre = [0] + list(accumulate(stones))

    @lru_cache(maxsize=None)
    def dfs(i, j):
        if i >= j:
            return 0
        take_left = pre[j + 1] - pre[i + 1] - dfs(i + 1, j)
        take_right = pre[j] - pre[i] - dfs(i, j - 1)
        return max(take_left, take_right)

    ans = dfs(0, n - 1)
    dfs.cache_clear()
    return ans

for stones, want in examples:
    print(stone_vii_memo(stones), "kỳ vọng", want)

6 kỳ vọng 6
122 kỳ vọng 122


**Độ phức tạp:** `n²/2` trạng thái, mỗi trạng thái `O(1)` ⇒ Time `O(n²)`, Space `O(n²)` cho cache + `O(n)` stack. Với `n = 1000` có thể vượt độ sâu đệ quy mặc định và chạy chậm do overhead gọi hàm — không nên nộp bản này nếu chưa tăng `setrecursionlimit`.

Code nộp (tham khảo — cần tăng giới hạn đệ quy):

```python
class Solution:
    def stoneGameVII(self, stones: List[int]) -> int:
        import sys
        sys.setrecursionlimit(10000)
        pre = [0] + list(accumulate(stones))
        @lru_cache(maxsize=None)
        def dfs(i, j):
            if i >= j:
                return 0
            return max(pre[j + 1] - pre[i + 1] - dfs(i + 1, j),
                       pre[j] - pre[i] - dfs(i, j - 1))
        ans = dfs(0, len(stones) - 1)
        dfs.cache_clear()
        return ans
```

### 7.2 Cách 3 — Bottom-up, bảng 2 chiều

Từng đoạn:
1. `dp = [[0]*n for _ in range(n)]` (đường chéo `dp[i][i] = 0`).
2. Lặp `i` từ `n−2` xuống 0, `j` từ `i+1` đến `n−1`.
3. Áp công thức với tổng tiền tố.

In [10]:
def stone_vii_table(stones):
    n = len(stones)
    pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            a = pre[j + 1] - pre[i + 1] - dp[i + 1][j]
            b = pre[j] - pre[i] - dp[i][j - 1]
            dp[i][j] = a if a > b else b
    return dp[0][n - 1]

for stones, want in examples:
    print(stone_vii_table(stones), "kỳ vọng", want)

6 kỳ vọng 6
122 kỳ vọng 122


**Độ phức tạp:** Time `O(n²)`, Space `O(n²)` (1000 × 1000 = 10⁶ số nguyên — chấp nhận được).

### 7.3 Cách 4 — Mảng 1 chiều cuộn, Space `O(n)`

Khi lặp `i` giảm dần, hàng `i` chỉ cần hàng `i+1` và các ô bên trái của chính hàng `i`. Dùng **một** mảng `dp` độ dài `n`:
- trước khi ghi `dp[j]` ở hàng `i`, `dp[j]` vẫn đang giữ giá trị hàng `i+1` (tức `dp[i+1][j]`);
- `dp[j−1]` đã được ghi ở hàng `i` (tức `dp[i][j−1]`).

Đúng hai giá trị ta cần.

In [11]:
def stone_vii_1d(stones):
    n = len(stones)
    pre = [0] + list(accumulate(stones))
    dp = [0] * n
    for i in range(n - 2, -1, -1):
        dp[i] = 0
        pi, pi1 = pre[i], pre[i + 1]
        for j in range(i + 1, n):
            a = pre[j + 1] - pi1 - dp[j]        # dp[j] lúc này = dp[i+1][j]
            b = pre[j] - pi - dp[j - 1]         # dp[j-1] lúc này = dp[i][j-1]
            dp[j] = a if a > b else b
    return dp[n - 1]

for stones, want in examples:
    print(stone_vii_1d(stones), "kỳ vọng", want)

6 kỳ vọng 6
122 kỳ vọng 122


**Độ phức tạp:** Time `O(n²)`, Space `O(n)`. Đây là bản nộp ở mục 9.

## 8. TRACE từng bước

Bảng `dp[i][j]` cho ví dụ 1 `stones = [5,3,1,4,2]`, `pre = [0,5,8,9,13,15]`. Đường chéo bằng 0.

| i \ j | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| 0 | 0 | 5 | 3 | 7 | **6** |
| 1 | · | 0 | 3 | 1 | 7 |
| 2 | · | · | 0 | 4 | 2 |
| 3 | · | · | · | 0 | 4 |
| 4 | · | · | · | · | 0 |

Ví dụ ô `dp[0][4]`: bỏ trái (viên 5) được `S(1,4) = 10`, trừ `dp[1][4] = 7` ⇒ `3`; bỏ phải (viên 2) được `S(0,3) = 13`, trừ `dp[0][3] = 7` ⇒ `6`. Lấy max ⇒ `6` — Alice nên bỏ viên 2 trước, đúng như lời giải mẫu của đề.

Ô `dp[1][3]` (đoạn `[3,1,4]`): bỏ 3 được `5`, trừ `dp[2][3] = 4` ⇒ `1`; bỏ 4 được `4`, trừ `dp[1][2] = 3` ⇒ `1`. Hai nước cho cùng kết quả.

In [12]:
def trace_table(stones):
    n = len(stones)
    pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            a = pre[j + 1] - pre[i + 1] - dp[i + 1][j]
            b = pre[j] - pre[i] - dp[i][j - 1]
            dp[i][j] = max(a, b)
            print(f"dp[{i}][{j}]: bỏ trái {pre[j+1]-pre[i+1]:3d} - dp[{i+1}][{j}]={dp[i+1][j]:3d} = {a:3d} | "
                  f"bỏ phải {pre[j]-pre[i]:3d} - dp[{i}][{j-1}]={dp[i][j-1]:3d} = {b:3d} -> {dp[i][j]}")
    print("\nbảng dp:")
    for row in dp:
        print(" ".join(f"{v:4d}" for v in row))
    return dp[0][n - 1]

trace_table([5, 3, 1, 4, 2])

dp[3][4]: bỏ trái   2 - dp[4][4]=  0 =   2 | bỏ phải   4 - dp[3][3]=  0 =   4 -> 4
dp[2][3]: bỏ trái   4 - dp[3][3]=  0 =   4 | bỏ phải   1 - dp[2][2]=  0 =   1 -> 4
dp[2][4]: bỏ trái   6 - dp[3][4]=  4 =   2 | bỏ phải   5 - dp[2][3]=  4 =   1 -> 2
dp[1][2]: bỏ trái   1 - dp[2][2]=  0 =   1 | bỏ phải   3 - dp[1][1]=  0 =   3 -> 3
dp[1][3]: bỏ trái   5 - dp[2][3]=  4 =   1 | bỏ phải   4 - dp[1][2]=  3 =   1 -> 1
dp[1][4]: bỏ trái   7 - dp[2][4]=  2 =   5 | bỏ phải   8 - dp[1][3]=  1 =   7 -> 7
dp[0][1]: bỏ trái   3 - dp[1][1]=  0 =   3 | bỏ phải   5 - dp[0][0]=  0 =   5 -> 5
dp[0][2]: bỏ trái   4 - dp[1][2]=  3 =   1 | bỏ phải   8 - dp[0][1]=  5 =   3 -> 3
dp[0][3]: bỏ trái   8 - dp[1][3]=  1 =   7 | bỏ phải   9 - dp[0][2]=  3 =   6 -> 7
dp[0][4]: bỏ trái  10 - dp[1][4]=  7 =   3 | bỏ phải  13 - dp[0][3]=  7 =   6 -> 6

bảng dp:
   0    5    3    7    6
   0    0    3    1    7
   0    0    0    4    2
   0    0    0    0    4
   0    0    0    0    0


6

Bảng in ra khớp bảng tính tay ở trên.

Trace thêm: **khôi phục nước đi tối ưu** từ bảng `dp` để thấy ván đấu cụ thể.

In [13]:
def replay(stones):
    n = len(stones)
    pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            dp[i][j] = max(pre[j + 1] - pre[i + 1] - dp[i + 1][j], pre[j] - pre[i] - dp[i][j - 1])
    i, j, turn, score = 0, n - 1, 0, [0, 0]
    while i < j:
        a = pre[j + 1] - pre[i + 1] - dp[i + 1][j]
        b = pre[j] - pre[i] - dp[i][j - 1]
        who = "Alice" if turn == 0 else "Bob  "
        if a >= b:
            gain = pre[j + 1] - pre[i + 1]; print(f"{who} bỏ trái {stones[i]:4d}, được {gain}"); i += 1
        else:
            gain = pre[j] - pre[i]; print(f"{who} bỏ phải {stones[j]:4d}, được {gain}"); j -= 1
        score[turn] += gain; turn ^= 1
    print(f"Alice={score[0]}, Bob={score[1]}, hiệu={score[0] - score[1]} (dp={dp[0][n-1]})")

replay([5, 3, 1, 4, 2])

Alice bỏ phải    2, được 13
Bob   bỏ trái    5, được 8
Alice bỏ trái    3, được 5
Bob   bỏ trái    1, được 4
Alice=18, Bob=12, hiệu=6 (dp=6)


In [14]:
replay([7, 90, 5, 1, 100, 10, 10, 2])

Alice bỏ phải    2, được 223
Bob   bỏ trái    7, được 216
Alice bỏ trái   90, được 126
Bob   bỏ trái    5, được 121
Alice bỏ trái    1, được 120
Bob   bỏ trái  100, được 20
Alice bỏ trái   10, được 10
Alice=479, Bob=357, hiệu=122 (dp=122)


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def stoneGameVII(self, stones: List[int]) -> int:
        n = len(stones)
        pre = [0]
        for x in stones:
            pre.append(pre[-1] + x)
        dp = [0] * n
        for i in range(n - 2, -1, -1):
            dp[i] = 0
            pi, pi1 = pre[i], pre[i + 1]
            for j in range(i + 1, n):
                a = pre[j + 1] - pi1 - dp[j]
                b = pre[j] - pi - dp[j - 1]
                dp[j] = a if a > b else b
        return dp[n - 1]

sol = Solution()
for stones, want in examples:
    got = sol.stoneGameVII(stones)
    print(got, "OK" if got == want else "SAI")

6 OK
122 OK


## 10. Vì sao thuật toán đúng

**Mệnh đề:** với mọi `i ≤ j`, `dp[i][j]` bằng hiệu lớn nhất (người đi − người kia) mà người đi bảo đảm được trên đoạn `stones[i..j]` khi cả hai chơi tối ưu.

**Chứng minh quy nạp theo độ dài `L = j − i + 1`:**
- `L = 1`: bỏ viên duy nhất được 0, không còn lượt ⇒ hiệu 0 = `dp[i][i]`. ✓
- `L ≥ 2`: người đi có đúng hai nước. Nếu bỏ trái, nhận `S(i+1, j)` và trao lượt trên đoạn `[i+1, j]`; theo giả thiết quy nạp, đối thủ đạt hiệu tối ưu `dp[i+1][j]` **theo góc nhìn đối thủ** ⇒ hiệu theo góc nhìn người đi là `S(i+1, j) − dp[i+1][j]`. Tương tự bỏ phải. Người đi chọn nước lớn hơn ⇒ đúng công thức. ✓

**Vì sao mục tiêu của Bob khớp:** Bob "tối thiểu hóa `A − B`" ⇔ "tối đa hóa `B − A`", đúng là hiệu theo góc nhìn người đi khi đến lượt Bob. Do đó `dp[0][n−1]` (Alice đi trước) là đáp án.

**Cách 4 đúng** vì tại thời điểm ghi `dp[j]` trong hàng `i`, ô `dp[j]` chưa bị ghi đè trong hàng này (vẫn là hàng `i+1`), còn `dp[j−1]` đã ghi đè (hàng `i`) — đúng hai phụ thuộc của công thức.

## 11. Phân tích Time / Space

Gọi `n = len(stones) ≤ 1000`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy thuần | `O(n · 2ⁿ)` | `O(n²)` stack | chỉ `n ≤ ~18` |
| 2. Memo top-down | `O(n²)` | `O(n²)` | đệ quy sâu `n`, overhead lớn |
| 3. Bảng 2D | `O(n²)` | `O(n²)` | ~5·10⁵ ô cần tính |
| 4. Mảng 1D | `O(n²)` | `O(n)` | **nộp bài** |

Số phép tính chính xác: `n(n−1)/2 ≈ 5·10⁵` với `n = 1000` — Python chạy dưới 1 giây.

In [16]:
random.seed(3)
big = [random.randint(1, 1000) for _ in range(1000)]
for name, fn in [("table", stone_vii_table), ("1d", stone_vii_1d), ("submit", Solution().stoneGameVII)]:
    t0 = time.perf_counter(); r = fn(big)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.3f}s")

table : 260555  0.080s


1d    : 260555  0.055s


submit: 260555  0.054s


## 12. Bộ test

Ví dụ đề, edge case (`n = 2`, mọi giá trị bằng nhau, giá trị lớn nhất), đối chiếu ngẫu nhiên với đệ quy thuần (seed cố định, `2 ≤ n ≤ 12`, `1 ≤ stones[i] ≤ 1000`) và giữa các cách `O(n²)` với `n` lớn hơn.

In [17]:
fns = [stone_vii_brute, stone_vii_memo, stone_vii_table, stone_vii_1d, lambda s: Solution().stoneGameVII(s)]

for stones, want in examples:
    for fn in fns:
        assert fn(stones) == want

edge = [([1, 2], 2), ([2, 1], 2), ([1000, 1000], 1000), ([1, 1, 1], 1)]
for stones, want in edge:
    assert stone_vii_brute(stones) == want, stones
    for fn in fns:
        assert fn(stones) == want

random.seed(2025)
for _ in range(400):
    n = random.randint(2, 12)
    st = [random.randint(1, random.choice([5, 1000])) for _ in range(n)]
    want = stone_vii_brute(st)
    for fn in fns[1:]:
        assert fn(st) == want, st
print("ví dụ + edge + 400 bộ ngẫu nhiên khớp brute force")

ví dụ + edge + 400 bộ ngẫu nhiên khớp brute force


In [18]:
random.seed(77)
for _ in range(30):
    n = random.randint(50, 300)
    st = [random.randint(1, 1000) for _ in range(n)]
    r = stone_vii_table(st)
    assert r == stone_vii_1d(st) == Solution().stoneGameVII(st)
    assert r >= 0          # người đi trước luôn không thua (Bob "luôn thua")
print("các cách O(n^2) khớp nhau trên n tới 300; hiệu luôn ≥ 0")

các cách O(n^2) khớp nhau trên n tới 300; hiệu luôn ≥ 0


Kiểm tra `r ≥ 0` minh họa nhận xét của đề "Bob luôn thua (hoặc hòa)": thực tế người đi trước luôn bảo đảm hiệu không âm. (Hãy thử chứng minh ở Bài 5.)

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam: bỏ viên nhỏ hơn ở hai đầu

Giữ lại tổng lớn nhất ngay lượt này, nhưng bỏ qua ảnh hưởng tới các lượt sau.

In [19]:
def wrong_greedy(stones):
    arr = list(stones); score = [0, 0]; turn = 0
    while len(arr) > 1:
        if arr[0] <= arr[-1]:
            arr.pop(0)
        else:
            arr.pop()
        score[turn] += sum(arr); turn ^= 1
    return score[0] - score[1]

random.seed(1)
for _ in range(2000):
    st = [random.randint(1, 10) for _ in range(random.randint(2, 7))]
    if wrong_greedy(st) != stone_vii_brute(st):
        print("phản ví dụ:", st, "| tham lam =", wrong_greedy(st), "| đúng =", stone_vii_brute(st))
        break

phản ví dụ: [5, 1, 7, 9, 2] | tham lam = 12 | đúng = 10


Chú ý: con số tham lam in ra có thể **lớn hơn** đáp án đúng — vì trong mô phỏng tham lam, Bob cũng chơi tham lam (không tối ưu), nên Alice "được lợi" một cách không thực tế. Đáp án của đề là giá trị khi **cả hai** chơi tối ưu; nếu Alice chơi tham lam còn Bob chơi tối ưu, Alice có thể nhận hiệu **nhỏ hơn** `dp`. Dù lệch theo chiều nào, tham lam không cho giá trị trò chơi.

**Sửa:** DP khoảng xét cả hai lựa chọn và hệ quả về sau.

### Lỗi 2 — Viết DP cộng điểm mà quên trừ phần của đối thủ

`dp[i][j] = max(S(i+1,j) + dp[i+1][j], …)` tính tổng điểm của **cả hai**, không phải hiệu.

In [20]:
def wrong_plus(stones):
    n = len(stones); pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            dp[i][j] = max(pre[j + 1] - pre[i + 1] + dp[i + 1][j], pre[j] - pre[i] + dp[i][j - 1])
    return dp[0][n - 1]

for stones, want in examples:
    print("sai =", wrong_plus(stones), "| đúng =", want)

sai = 35 | đúng = 6
sai = 1148 | đúng = 122


**Sửa:** dấu **trừ** — `S − dp[đoạn con]`.

### Lỗi 3 — Sai thứ tự lặp: `i` tăng dần

Khi `i` tăng, `dp[i+1][j]` chưa được tính (vẫn là 0) ⇒ kết quả sai.

In [21]:
def wrong_order(stones):
    n = len(stones); pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    for i in range(n):                 # SAI: phải là n-2 -> 0
        for j in range(i + 1, n):
            dp[i][j] = max(pre[j + 1] - pre[i + 1] - dp[i + 1][j], pre[j] - pre[i] - dp[i][j - 1])
    return dp[0][n - 1]

for stones, want in examples:
    print("sai =", wrong_order(stones), "| đúng =", want)

sai = 10 | đúng = 6
sai = 218 | đúng = 122


**Sửa:** lặp `i` từ `n−2` về `0` (hoặc theo độ dài đoạn tăng dần).

### Lỗi 4 — Tổng tiền tố lệch chỉ số

Dùng `pre[j] − pre[i]` cho "tổng `i+1..j`" (thực chất là tổng `i..j−1`). Lỗi off-by-one rất hay gặp.

In [22]:
def wrong_prefix(stones):
    n = len(stones); pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            left = pre[j] - pre[i + 1]       # SAI: thiếu stones[j]
            right = pre[j] - pre[i]
            dp[i][j] = max(left - dp[i + 1][j], right - dp[i][j - 1])
    return dp[0][n - 1]

for stones, want in examples:
    print("sai =", wrong_prefix(stones), "| đúng =", want)

sai = 7 | đúng = 6
sai = 122 | đúng = 122


**Sửa:** tổng `stones[l..r]` = `pre[r+1] − pre[l]`. Luôn kiểm tra công thức trên một đoạn nhỏ (mục 3.1).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Dấu hiệu DP khoảng:** chỉ được lấy **đầu hoặc cuối** dãy ⇒ phần còn lại luôn là đoạn liên tiếp `[i, j]`.
- **Trò chơi hai người tối ưu** ⇒ định nghĩa hiệu theo góc nhìn người đi, công thức `max(gain − dp[next])`.
- Điểm phụ thuộc tổng đoạn ⇒ chuẩn bị **tổng tiền tố**.
- Python + `n = 1000` + `O(n²)` ⇒ ưu tiên bottom-up, biến cục bộ, tránh `max()` trong vòng lặp nóng (dùng `a if a > b else b`).
- Cùng họ: Stone Game I (877), Predict the Winner (486) — gần như cùng khung, chỉ khác điểm nhận được.

## 15. Biến thể của bài

1. **Predict the Winner (486):** điểm = giá trị viên bị bỏ (không phải phần còn lại) ⇒ `dp[i][j] = max(a[i] − dp[i+1][j], a[j] − dp[i][j−1])`.
2. **Stone Game (877):** như 486 nhưng `n` chẵn ⇒ người đi trước luôn thắng (có lập luận chẵn/lẻ).
3. **Bob đi trước:** đáp án vẫn là `dp[0][n−1]` nhưng mang nghĩa Bob − Alice.
4. **Trả về điểm của từng người:** `A + B` = tổng mọi điểm đã nhận (tính được khi khôi phục nước đi) và `A − B = dp` ⇒ giải hệ.
5. **Được bỏ tối đa `k` viên ở một đầu mỗi lượt:** trạng thái vẫn là `[i, j]`, mỗi trạng thái có `2k` lựa chọn.

In [23]:
def predict_winner_diff(nums):
    """Biến thể 486: điểm = giá trị viên bị lấy."""
    n = len(nums); dp = [0] * n
    for i in range(n - 1, -1, -1):
        dp[i] = nums[i]
        for j in range(i + 1, n):
            dp[j] = max(nums[i] - dp[j], nums[j] - dp[j - 1])
    return dp[n - 1]

print("486 [1,5,2] ->", predict_winner_diff([1, 5, 2]), "(âm => người đi trước thua)")
print("486 [1,5,233,7] ->", predict_winner_diff([1, 5, 233, 7]))

486 [1,5,2] -> -2 (âm => người đi trước thua)
486 [1,5,233,7] -> 222


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | thử mọi nước | `O(n·2ⁿ)` | `O(n²)` | kiểm chứng `n` nhỏ |
| 2. Memo `(i, j)` | ghi nhớ đoạn | `O(n²)` | `O(n²)` | viết nhanh khi phỏng vấn, `n` vừa |
| 3. Bảng 2D | bottom-up theo `i` giảm | `O(n²)` | `O(n²)` | dễ debug, in được bảng |
| 4. Mảng 1D | cuộn hàng | `O(n²)` | `O(n)` | **nộp bài**, tiết kiệm bộ nhớ |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `stoneGameVII([3, 8])` và `stoneGameVII([4, 4, 4])` bằng bao nhiêu?

**Bài 2 (tính tay):** lập bảng `dp` cho `stones = [2, 7, 4]`.

**Bài 3 (điền TODO):** hoàn thiện `stone_vii_by_length(stones)` lặp theo **độ dài đoạn** `L = 2..n`.

**Bài 4 (tự viết lại):** viết hàm `scores(stones)` trả về cặp `(điểm Alice, điểm Bob)` khi chơi tối ưu (dùng khôi phục nước đi).

**Bài 5 (mini challenge):** chứng minh rằng với mọi đoạn, `0 ≤ dp[i][j] ≤ S(i, j)` — từ đó suy ra người đi trước không bao giờ thua (đúng như câu "Bob luôn thua" của đề). Kiểm chứng bằng máy trên input ngẫu nhiên.

In [24]:
def stone_vii_by_length(stones):
    n = len(stones)
    pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    # TODO: for L in range(2, n + 1): for i in range(0, n - L + 1): j = i + L - 1 ...
    return dp[0][n - 1]

## 18. Gợi ý 3 mức

**Bài 1:** (1) Đoạn 2 viên ⇒ `max(a, b)`. (2) `[4,4,4]`: Alice được 8, Bob được 4, Alice được 0. (3) Đáp án `8` và `4`.

**Bài 2:** (1) `dp[0][1] = max(2, 7) = 7`, `dp[1][2] = max(7, 4) = 7`. (2) `dp[0][2] = max(S(1,2) − dp[1][2], S(0,1) − dp[0][1]) = max(11 − 7, 9 − 7)`. (3) `= 4`.

**Bài 3:** (1) `j = i + L − 1`. (2) Công thức không đổi. (3) Độ dài tăng dần bảo đảm đoạn con đã có.

**Bài 4:** (1) Tính bảng `dp` trước. (2) Mô phỏng như `replay`, cộng điểm vào người đang đi. (3) Kiểm tra `A − B == dp[0][n−1]`.

**Bài 5:** (1) Quy nạp theo độ dài đoạn, chứng minh **đồng thời** hai bất đẳng thức. (2) Cận trên: mỗi nhánh là `S(đoạn con) − dp(đoạn con)`, mà `dp(đoạn con) ≥ 0`. (3) Cận dưới: nhánh bỏ trái là `S(i+1, j) − dp[i+1][j]`, mà `dp[i+1][j] ≤ S(i+1, j)`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().stoneGameVII([3, 8]) == 8 and Solution().stoneGameVII([4, 4, 4]) == 4

# Bài 2
assert stone_vii_table([2, 7, 4]) == 4

# Bài 3
def stone_vii_by_length_ans(stones):
    n = len(stones); pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    for L in range(2, n + 1):
        for i in range(0, n - L + 1):
            j = i + L - 1
            dp[i][j] = max(pre[j + 1] - pre[i + 1] - dp[i + 1][j], pre[j] - pre[i] - dp[i][j - 1])
    return dp[0][n - 1]

# Bài 4
def scores(stones):
    n = len(stones); pre = [0] + list(accumulate(stones))
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            dp[i][j] = max(pre[j + 1] - pre[i + 1] - dp[i + 1][j], pre[j] - pre[i] - dp[i][j - 1])
    i, j, t, sc = 0, n - 1, 0, [0, 0]
    while i < j:
        a = pre[j + 1] - pre[i + 1] - dp[i + 1][j]
        b = pre[j] - pre[i] - dp[i][j - 1]
        if a >= b:
            sc[t] += pre[j + 1] - pre[i + 1]; i += 1
        else:
            sc[t] += pre[j] - pre[i]; j -= 1
        t ^= 1
    return tuple(sc)

random.seed(9)
for _ in range(300):
    st = [random.randint(1, 1000) for _ in range(random.randint(2, 40))]
    assert stone_vii_by_length_ans(st) == stone_vii_1d(st)
    A, B = scores(st)
    assert A - B == stone_vii_1d(st)
assert scores([5, 3, 1, 4, 2]) == (18, 12)
print("Bài 1-4: OK")

Bài 1-4: OK


In [26]:
# Bài 5 — kiểm chứng 0 <= dp[i][j] <= S(i..j) cho mọi đoạn (chứng minh ở cell markdown bên dưới)
random.seed(10)
for _ in range(300):
    st = [random.randint(1, 50) for _ in range(random.randint(2, 30))]
    n = len(st); pre = [0] + list(accumulate(st))
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 2, -1, -1):
        for j in range(i + 1, n):
            dp[i][j] = max(pre[j + 1] - pre[i + 1] - dp[i + 1][j], pre[j] - pre[i] - dp[i][j - 1])
            assert 0 <= dp[i][j] <= pre[j + 1] - pre[i]
print("Bài 5: 0 <= dp[i][j] <= S(i..j) với mọi đoạn đã thử")

Bài 5: 0 <= dp[i][j] <= S(i..j) với mọi đoạn đã thử


Lập luận Bài 5 hoàn chỉnh: chứng minh đồng thời `0 ≤ dp[i][j] ≤ S(i, j)` bằng quy nạp. Cận trên: mỗi nhánh có dạng `S(đoạn con) − dp(đoạn con) ≤ S(đoạn con) ≤ S(i, j)` (vì `dp ≥ 0`). Cận dưới: nhánh bỏ trái cho `S(i+1, j) − dp[i+1][j] ≥ S(i+1,j) − S(i+1,j) = 0`.

## 20. Template nhớ nhanh + Checklist

```python
# DP khoảng cho trò chơi "lấy đầu/cuối", hiệu theo góc nhìn người đi
pre = [0] + list(accumulate(a))
dp = [0] * n
for i in range(n - 2, -1, -1):
    for j in range(i + 1, n):
        left  = (pre[j + 1] - pre[i + 1]) - dp[j]       # dp[j]   = dp[i+1][j]
        right = (pre[j] - pre[i]) - dp[j - 1]           # dp[j-1] = dp[i][j-1]
        dp[j] = max(left, right)
return dp[n - 1]
```

**Checklist trước khi submit:**
- [ ] Điểm nhận = tổng **phần còn lại**, không phải viên bị bỏ.
- [ ] Công thức có **dấu trừ** `− dp[đoạn con]`.
- [ ] Tổng tiền tố đúng chỉ số: `S(l, r) = pre[r+1] − pre[l]`.
- [ ] Lặp `i` **giảm dần**, `j` **tăng dần** (với mảng 1D phải đúng thứ tự này).
- [ ] Không dùng đệ quy sâu `n = 1000` mà quên giới hạn đệ quy.
- [ ] Đã thử `n = 2`.